In [ ]:
!pip install ultralytics
!pip install matplotlib opencv-python numpy pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/TCC/Modelo/Result 1024/Single CLS (BEST)/weights/best.pt')

In [ ]:
img_path = '/content/drive/MyDrive/506_crop_2_4.jpg'
txt_path = '/content/runs/detect/predict/labels/506_crop_2_4.txt'

In [ ]:
results_img = model.predict(source= img_path ,
                            conf = 0.8,
                            save = True,
                            save_txt=True,
                            project= 'runs/detect')

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

def parse_linha(linha, img_size=1024):
    partes = list(map(float, linha.strip().split()))
    coords = partes[1:]  # ignora a classe (partes[0])
    pontos = [(coords[i] * img_size, coords[i+1] * img_size) for i in range(0, len(coords), 2)]
    return pontos

def salvar_linha_segmentacao_grafico(pontos, idx, pasta_linhas="telhados_linhas"):
    os.makedirs(pasta_linhas, exist_ok=True)

    xs, ys = zip(*pontos)
    plt.figure(figsize=(6,6))
    plt.plot(xs, ys, color='green', linewidth=2, label='Área útil do telhado')
    plt.xlabel('Metros')
    plt.ylabel('Metros')
    plt.grid(True)
    plt.legend(loc='upper right')
    plt.axis('equal')

    # Salvar imagem no estilo do gráfico
    out_linha_path = f"{pasta_linhas}/linha_{idx+1}.png"
    plt.savefig(out_linha_path, bbox_inches='tight', pad_inches=0.1, transparent=False)
    plt.close()

def calcular_area_e_recortar(img_path, txt_path, resolucao=0.1, img_size=1024, pasta_saida="telhados_recortados", pasta_linhas="telhados_linhas"):
    os.makedirs(pasta_saida, exist_ok=True)
    os.makedirs(pasta_linhas, exist_ok=True)

    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    h, w = img.shape[:2]

    areas = []
    telhados_recortes = []

    with open(txt_path, 'r') as f:
        for idx, linha in enumerate(f):
            pontos = parse_linha(linha, img_size)

            # Calcular área
            area_pixels = calcular_area_poligono(pontos)
            area_metros = area_pixels * (resolucao ** 2)
            areas.append(area_metros)

            # Criar máscara e recorte (mantém igual ao seu código)
            mask = np.zeros((h, w), dtype=np.uint8)
            pts = np.array(pontos, np.int32).reshape((-1, 1, 2))
            cv2.fillPoly(mask, [pts], 255)
            telhado_recorte = cv2.bitwise_and(img, img, mask=mask)
            x, y, w_box, h_box = cv2.boundingRect(pts)
            telhado_cortado = img[y:y+h_box, x:x+w_box]

            # Salvar telhado recortado
            out_path = f"{pasta_saida}/telhado_{idx+1}.jpg"
            cv2.imwrite(out_path, telhado_cortado)
            telhados_recortes.append((telhado_cortado, idx+1, area_metros))

            # Salvar linha da segmentação como gráfico
            salvar_linha_segmentacao_grafico(pontos, idx, pasta_linhas=pasta_linhas)

    return areas, telhados_recortes

def calcular_area_poligono(pontos):
    n = len(pontos)
    area = 0.0
    for i in range(n):
        x1, y1 = pontos[i]
        x2, y2 = pontos[(i + 1) % n]
        area += (x1 * y2) - (x2 * y1)
    return abs(area) / 2.0

areas_telhados, recortes = calcular_area_e_recortar(img_path, txt_path)

# Mostrar no notebook
for telhado, idx, area in recortes:
    plt.figure(figsize=(4, 4))
    plt.imshow(cv2.cvtColor(telhado, cv2.COLOR_BGR2RGB))
    plt.title(f"Telhado {idx} - {area:.2f} m²")
    plt.axis("off")
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import numpy as np
import os

def parse_linha(linha, img_size=1024):
    partes = list(map(float, linha.strip().split()))
    coords = partes[1:]
    pontos = [(coords[i] * img_size, coords[i+1] * img_size) for i in range(0, len(coords), 2)]
    return pontos

def plot_segmentacao(pontos, idx, pasta_saida="plots_telhados"):
    os.makedirs(pasta_saida, exist_ok=True)
    xs, ys = zip(*pontos)
    xs_m = np.array(xs) * resolucao
    ys_m = np.array(ys) * resolucao

    plt.figure(figsize=(6,6))
    plt.plot(xs_m, ys_m, color='green', linewidth=2)
    plt.axis('equal')
    plt.axis('off')  # remove eixos e bordas

    out_path = f"{pasta_saida}/segmentacao_{idx+1}.png"
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0, transparent=False, facecolor='white')
    plt.close()


def plot_segmentacao_e_area_util(pontos, poligono_interno, idx, pasta_saida="plots_telhados"):
    os.makedirs(pasta_saida, exist_ok=True)
    xs, ys = zip(*pontos)
    xs_m = np.array(xs) * resolucao
    ys_m = np.array(ys) * resolucao

    plt.figure(figsize=(6,6))
    plt.plot(xs_m, ys_m, color='green', linewidth=2)

    # Plota área útil se existir
    if not poligono_interno.is_empty and poligono_interno.is_valid:
        if poligono_interno.geom_type == 'Polygon':
            xs_in, ys_in = poligono_interno.exterior.xy
            xs_in = np.array(xs_in) * resolucao
            ys_in = np.array(ys_in) * resolucao
            plt.plot(xs_in, ys_in, color='orange', linewidth=2)
        elif poligono_interno.geom_type == 'MultiPolygon':
            for poly in poligono_interno.geoms:
                xs_in, ys_in = poly.exterior.xy
                xs_in = np.array(xs_in) * resolucao
                ys_in = np.array(ys_in) * resolucao
                plt.plot(xs_in, ys_in, color='orange', linewidth=2)

    plt.axis('equal')
    plt.axis('off')

    out_path = f"{pasta_saida}/segmentacao_area_util_{idx+1}.png"
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0, transparent=False, facecolor='white')
    plt.close()

# Parâmetros globais
img_size = 1024
resolucao = 0.1
margem_metros = 0.5

# Exemplo de uso:
caminho = txt_path
with open(caminho) as f:
    for idx, linha in enumerate(f):
        pontos = parse_linha(linha, img_size)
        poligono = Polygon(pontos)
        poligono_interno = poligono.buffer(-margem_metros / resolucao)
        plot_segmentacao(pontos, idx)
        plot_segmentacao_e_area_util(pontos, poligono_interno, idx)

In [ ]:
from shapely.geometry import Polygon, box
from shapely import affinity
import matplotlib.pyplot as plt

# --- CONFIGURAÇÕES ---
resolucao = 0.1  # metros por pixel
largura_painel = 2.465  # metros
altura_painel = 1.134   # metros
margem_metros = 0.5     # margem para área útil
arquivo_txt = txt_path  # caminho do arquivo de segmentação

# parâmetros do painel
potencia_painel_kw = 0.560
irradiacao_media = 5.094
performance_ratio = 0.8     # perdas (80% eficiência)

# --- FUNÇÕES ---
def parse_linha(linha, img_size=1024):
    partes = list(map(float, linha.strip().split()))
    coords = partes[1:]
    pontos = [(coords[i] * img_size, coords[i+1] * img_size) for i in range(0, len(coords), 2)]
    return pontos

def contar_paineis_rotacionados(poligono, largura, altura, resolucao, angulo):
    poligono_metros = affinity.scale(poligono, xfact=resolucao, yfact=resolucao, origin=(0, 0))
    minx, miny, maxx, maxy = poligono_metros.bounds

    painel_base = box(0, 0, largura, altura)
    painel_rot = affinity.rotate(painel_base, angulo, origin='center')
    pw, ph = painel_rot.bounds[2] - painel_rot.bounds[0], painel_rot.bounds[3] - painel_rot.bounds[1]

    desloc_x = pw * 0.1
    desloc_y = ph * 0.1

    paineis = []
    x = minx
    while x + pw <= maxx:
        y = miny
        while y + ph <= maxy:
            candidatos = [
                affinity.translate(painel_rot, xoff=x, yoff=y),
                affinity.translate(painel_rot, xoff=x+desloc_x, yoff=y),
                affinity.translate(painel_rot, xoff=x, yoff=y+desloc_y),
                affinity.translate(painel_rot, xoff=x+desloc_x, yoff=y+desloc_y),
            ]

            for candidato in candidatos:
                if poligono_metros.contains(candidato):
                    if all(candidato.intersection(p).area < 1e-6 for p in paineis):
                        paineis.append(candidato)
                        break
            y += ph
        x += pw

    return paineis, poligono_metros

# --- PROCESSAMENTO ---
angulos = [0, 45, 90, 135]

with open(arquivo_txt) as f:
    for i, linha in enumerate(f):
        if i > 0:
          break
        pontos = parse_linha(linha)
        poligono = Polygon(pontos)
        margem_pixels = margem_metros / resolucao
        poligono_util = poligono.buffer(-margem_pixels)

        if poligono_util.is_valid and not poligono_util.is_empty:
            resultados = []

            # testa cada ângulo
            for angulo in angulos:
                paineis, poligono_metros = contar_paineis_rotacionados(
                    poligono_util, largura_painel, altura_painel, resolucao, angulo
                )
                resultados.append((len(paineis), angulo, paineis, poligono_metros))

            # escolhe o melhor resultado
            melhor = max(resultados, key=lambda x: x[0])
            qtd, angulo_usado, paineis, poligono_metros = melhor

            # cálculo do potencial solar
            geracao_diaria_kw = qtd * potencia_painel_kw * irradiacao_media * performance_ratio
            geracao_anual_kw = geracao_diaria_kw * 365

            # --- RESULTADO ---
            print(f"Telhado {i+1}: {qtd} painéis → {geracao_anual_kw:.2f} kWh/ano (ângulo {angulo_usado}°)")

            # --- PLOT ---
            fig, ax = plt.subplots(figsize=(8, 6))

            # contorno do telhado
            x, y = poligono_metros.exterior.xy
            ax.plot(x, y, color='orange', linewidth=2)

            # desenha os painéis
            for painel in paineis:
                x, y = painel.exterior.xy
                ax.fill(x, y, color='blue', alpha=0.6)

            ax.set_aspect('equal')
            ax.axis('off')
            plt.show()

        else:
            print(f"Telhado {i+1}: área útil inválida")


In [ ]:
from shapely.geometry import Polygon, box
from shapely import affinity
import matplotlib.pyplot as plt

# --- CONFIGURAÇÕES ---
resolucao = 0.1  # metros por pixel
largura_painel = 2.465  # metros
altura_painel = 1.134   # metros
margem_metros = 0.5     # margem para área útil
arquivo_txt = txt_path  # caminho do arquivo de segmentação

# parâmetros do painel
potencia_painel_kw = 0.560
irradiacao_media = 5.094
performance_ratio = 0.8     # perdas (80% eficiência)

# --- FUNÇÕES ---
def parse_linha(linha, img_size=1024):
    partes = list(map(float, linha.strip().split()))
    coords = partes[1:]
    pontos = [(coords[i] * img_size, coords[i+1] * img_size) for i in range(0, len(coords), 2)]
    return pontos

def contar_paineis_rotacionados(poligono, largura, altura, resolucao, angulo):
    poligono_metros = affinity.scale(poligono, xfact=resolucao, yfact=resolucao, origin=(0, 0))
    minx, miny, maxx, maxy = poligono_metros.bounds

    painel_base = box(0, 0, largura, altura)
    painel_rot = affinity.rotate(painel_base, angulo, origin='center')
    pw, ph = painel_rot.bounds[2] - painel_rot.bounds[0], painel_rot.bounds[3] - painel_rot.bounds[1]

    desloc_x = pw * 0.1
    desloc_y = ph * 0.1

    paineis = []
    x = minx
    while x + pw <= maxx:
        y = miny
        while y + ph <= maxy:
            candidatos = [
                affinity.translate(painel_rot, xoff=x, yoff=y),
                affinity.translate(painel_rot, xoff=x+desloc_x, yoff=y),
                affinity.translate(painel_rot, xoff=x, yoff=y+desloc_y),
                affinity.translate(painel_rot, xoff=x+desloc_x, yoff=y+desloc_y),
            ]

            for candidato in candidatos:
                if poligono_metros.contains(candidato):
                    if all(candidato.intersection(p).area < 1e-6 for p in paineis):
                        paineis.append(candidato)
                        break
            y += ph
        x += pw

    return paineis, poligono_metros

# --- PROCESSAMENTO ---
angulos = [0, 45, 90, 135]

with open(arquivo_txt) as f:
    for i, linha in enumerate(f):
        if i > 0:  # só processa o primeiro telhado
            break
        pontos = parse_linha(linha)
        poligono = Polygon(pontos)
        margem_pixels = margem_metros / resolucao
        poligono_util = poligono.buffer(-margem_pixels)

        if poligono_util.is_valid and not poligono_util.is_empty:
            resultados = []

            # testa cada ângulo
            for angulo in angulos:
                paineis, poligono_metros = contar_paineis_rotacionados(
                    poligono_util, largura_painel, altura_painel, resolucao, angulo
                )
                resultados.append((len(paineis), angulo, paineis, poligono_metros))

            # --- PRINT DE TODOS OS ÂNGULOS ---
            print(f"\nTelhado {i+1}:")
            for qtd, angulo, _, _ in resultados:
                print(f"  Ângulo {angulo}° → {qtd} painéis")

            # escolhe o melhor resultado
            melhor = max(resultados, key=lambda x: x[0])
            qtd, angulo_usado, paineis, poligono_metros = melhor

            # cálculo do potencial solar
            geracao_diaria_kw = qtd * potencia_painel_kw * irradiacao_media * performance_ratio
            geracao_anual_kw = geracao_diaria_kw * 365

            # --- RESULTADO FINAL ---
            print(f"  Melhor configuração: {qtd} painéis → {geracao_anual_kw:.2f} kWh/ano (ângulo {angulo_usado}°)")

            # --- PLOT DO MELHOR ÂNGULO ---
            fig, ax = plt.subplots(figsize=(8, 6))

            # contorno do telhado
            x, y = poligono_metros.exterior.xy
            ax.plot(x, y, color='orange', linewidth=2)

            # desenha os painéis
            for painel in paineis:
                x, y = painel.exterior.xy
                ax.fill(x, y, color='blue', alpha=0.6)

            ax.set_aspect('equal')
            ax.axis('off')
            plt.show()

        else:
            print(f"Telhado {i+1}: área útil inválida")
